# 🎙️ Zehon — voix off

**Une vidéo = un lancement.** Le notebook lit `02_script_voix.txt` dans le dossier Drive de la vidéo, le fait dire par ta voix clonée (Qwen3-TTS), et dépose dans `voix/` :

| Fichier | Contenu |
|---|---|
| `voix.wav` | la voix off complète |
| `mots.json` | l'horodatage de chaque phrase et de chaque mot (pour caler les images) |
| `etat.json` | l'état du run : `en_cours`, `pret` ou `echec`. C'est ce que lit la skill `content-maker` pour te dire si l'audio est prêt |

## Mode d'emploi

1. **Exécution → Modifier le type d'exécution → GPU T4.**
2. Remplis le formulaire de la cellule 1 (souvent : rien à changer).
3. **Exécution → Tout exécuter.** Autorise l'accès à Google Drive quand Colab le demande.
4. Première fois, ou nouvelle voix : coche `NOUVELLE_VOIX`, colle la **transcription exacte** de ton échantillon dans `TEXTE_REFERENCE`, puis choisis le fichier audio quand le bouton d'envoi apparaît. La voix est rangée dans `Zehon/Charte/voix/` et resservira aux vidéos suivantes.

Si Colab se coupe en route, relance **Tout exécuter** : les phrases déjà faites sont gardées sur Drive et ne sont pas refaites.

Pour refaire quelques phrases mal dites : mets leurs numéros dans `REFAIRE_LIGNES` (par exemple `12, 40`) et relance.

In [ ]:
# ── Cellule 1 : réglages ────────────────────────────────────────────────────
VIDEO = ""  # @param {type:"string"}
# Vide = la vidéo la plus récente qui a un 02_script_voix.txt et pas encore de voix prête.
# Sinon, le nom du dossier : "01_sel".

NOUVELLE_VOIX = False  # @param {type:"boolean"}
# Coché = tu envoies un nouvel échantillon de ta voix (5 à 15 s, une seule voix, sans musique).

TEXTE_REFERENCE = ""  # @param {type:"string"}
# La transcription EXACTE de l'échantillon, mot pour mot. Seulement si NOUVELLE_VOIX est coché.

REFAIRE_LIGNES = ""  # @param {type:"string"}
# Numéros de lignes de 02_script_voix.txt à regénérer (autre graine) : "12, 40". Vide = rien.

RACINE = "/content/drive/MyDrive/Zehon"  # @param {type:"string"}
# Le dossier Zehon dans ton Drive. Si Zehon est « partagé avec moi », ajoute-lui un raccourci dans « Mon Drive ».

GRAINE = 42        # graine de base : la phrase n utilise GRAINE + n (résultat reproductible)
PAUSE_MS = 300     # silence entre deux phrases
CIBLE_LUFS = -16.0 # volume cible (YouTube)
SEUIL_A_REECOUTER = 0.80  # une phrase transcrite à moins de 80 % du texte est signalée

In [ ]:
# ── Cellule 2 : installation et Drive ───────────────────────────────────────
import subprocess, sys
for paquet in ["qwen-tts", "faster-whisper", "librosa", "soundfile", "pyloudnorm"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet], capture_output=True, text=True)
    print(("✓ " if r.returncode == 0 else "✗ ") + paquet + ("" if r.returncode == 0 else "\n" + r.stderr[-500:]))

import os
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

import difflib, hashlib, json, re, shutil, time, unicodedata
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa
import pyloudnorm as pyln
import torch

RACINE = Path(RACINE)
if not RACINE.is_dir():
    candidats = [p for p in Path("/content/drive/MyDrive").glob("**/Zehon") if p.is_dir()][:3]
    raise SystemExit(f"❌ Dossier Zehon introuvable : {RACINE}\n   Candidats trouvés : {candidats or 'aucun'}\n"
                     "   → Corrige RACINE dans la cellule 1.")
print(f"✓ Zehon : {RACINE}")
if not torch.cuda.is_available():
    print("⚠️  Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4 (sinon, 5 à 10 fois plus lent).")

def maintenant():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

In [ ]:
# ── Cellule 3 : la vidéo et la voix de référence ────────────────────────────
VIDEOS = RACINE / "Videos"

def voix_prete(d):
    try:
        return json.loads((d / "voix" / "etat.json").read_text(encoding="utf-8")).get("statut") == "pret"
    except Exception:
        return False

if VIDEO:
    DOSSIER = VIDEOS / VIDEO
else:
    attente = [d for d in sorted(VIDEOS.iterdir()) if (d / "02_script_voix.txt").is_file() and not voix_prete(d)]
    if not attente:
        raise SystemExit("❌ Aucune vidéo n'attend sa voix (02_script_voix.txt présent et voix pas encore prête).\n"
                         "   → Pour refaire une voix déjà prête, écris le nom du dossier dans VIDEO.")
    DOSSIER = attente[-1]
SCRIPT = DOSSIER / "02_script_voix.txt"
if not SCRIPT.is_file():
    raise SystemExit(f"❌ {SCRIPT} introuvable : le script voix n'est pas encore écrit (étape 3b de content-maker).")
DOSSIER_VOIX = DOSSIER / "voix"
DOSSIER_VOIX.mkdir(exist_ok=True)
print(f"🎬 Vidéo : {DOSSIER.name}")

def ecrire_etat(**champs):
    chemin = DOSSIER_VOIX / "etat.json"
    try:
        etat = json.loads(chemin.read_text(encoding="utf-8"))
    except Exception:
        etat = {}
    etat.update(champs, maj=maintenant())
    tmp = chemin.with_suffix(".tmp")
    tmp.write_text(json.dumps(etat, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, chemin)

# La voix de référence vit dans Charte/voix/ : une fois envoyée, elle sert à toutes les vidéos.
CHARTE_VOIX = RACINE / "Charte" / "voix"
CHARTE_VOIX.mkdir(parents=True, exist_ok=True)
REF_WAV, REF_TXT = CHARTE_VOIX / "ref.wav", CHARTE_VOIX / "ref.txt"

if NOUVELLE_VOIX:
    if not TEXTE_REFERENCE.strip():
        raise SystemExit("❌ NOUVELLE_VOIX est coché mais TEXTE_REFERENCE est vide : colle la transcription exacte de l'échantillon.")
    from google.colab import files
    print("⬆️  Choisis le fichier audio de ta voix (wav, mp3, m4a…) :")
    envoi = files.upload()
    if not envoi:
        raise SystemExit("❌ Aucun fichier reçu.")
    nom, contenu = next(iter(envoi.items()))
    brut = Path("/content") / nom
    brut.write_bytes(contenu)
    audio, taux = librosa.load(str(brut), sr=24000, mono=True)
    audio, _ = librosa.effects.trim(audio, top_db=35)
    duree = len(audio) / taux
    if duree < 3:
        raise SystemExit(f"❌ Échantillon trop court ({duree:.1f} s) : il faut 5 à 15 s.")
    if duree > 20:
        print(f"⚠️  Échantillon long ({duree:.1f} s) : 5 à 15 s suffisent, et la transcription doit couvrir tout l'audio.")
    if REF_WAV.exists():  # on garde l'ancienne voix, datée, au lieu de l'écraser
        archive = CHARTE_VOIX / "anciennes"
        archive.mkdir(exist_ok=True)
        horodate = datetime.now().strftime("%Y%m%d_%H%M%S")
        shutil.move(str(REF_WAV), archive / f"{horodate}_ref.wav")
        if REF_TXT.exists():
            shutil.move(str(REF_TXT), archive / f"{horodate}_ref.txt")
    sf.write(REF_WAV, audio, taux)
    REF_TXT.write_text(TEXTE_REFERENCE.strip(), encoding="utf-8")
    print(f"✓ Nouvelle voix rangée dans Charte/voix/ ({duree:.1f} s)")

if not (REF_WAV.is_file() and REF_TXT.is_file()):
    raise SystemExit("❌ Aucune voix de référence dans Charte/voix/.\n"
                     "   → Coche NOUVELLE_VOIX, remplis TEXTE_REFERENCE et relance.")
TEXTE_REF = REF_TXT.read_text(encoding="utf-8").strip()
EMPREINTE_VOIX = hashlib.sha256(REF_WAV.read_bytes() + TEXTE_REF.encode("utf-8")).hexdigest()[:12]
print(f"✓ Voix de référence : Charte/voix/ref.wav ({EMPREINTE_VOIX})")

In [ ]:
# ── Cellule 4 : synthèse, phrase par phrase ─────────────────────────────────
phrases = [l.strip() for l in SCRIPT.read_text(encoding="utf-8").splitlines() if l.strip()]
a_refaire = {int(x) for x in re.findall(r"\d+", REFAIRE_LIGNES)}
print(f"📝 {len(phrases)} phrases" + (f", à refaire : {sorted(a_refaire)}" if a_refaire else ""))
ecrire_etat(statut="en_cours", video=DOSSIER.name, phrases=len(phrases), debut=maintenant(),
            voix=EMPREINTE_VOIX, erreur=None)

from qwen_tts import Qwen3TTSModel
from transformers.utils import logging as journal_hf
journal_hf.set_verbosity_error()  # masque « Setting pad_token_id… », répété à chaque phrase et sans conséquence
cuda = torch.cuda.is_available()
modele = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0" if cuda else "cpu",
    dtype=torch.bfloat16 if cuda else torch.float32,
)

# Le cache vit sur Drive : si Colab se coupe, le run suivant reprend là où il s'était arrêté.
CACHE = DOSSIER_VOIX / "cache"
CACHE.mkdir(exist_ok=True)

def graine(n):  # n commence à 1 ; une phrase « à refaire » change de graine, donc de nom de cache
    return GRAINE + n + (1000 if n in a_refaire else 0)

def nom_cache(n, phrase):
    cle = hashlib.sha256("\x00".join([phrase, str(graine(n)), EMPREINTE_VOIX]).encode("utf-8")).hexdigest()[:16]
    return f"{n:04d}_{cle}.wav"

attendus = {nom_cache(n, p) for n, p in enumerate(phrases, 1)}
for f in list(CACHE.glob("*.wav")) + list(CACHE.glob("*.tmp")):
    if f.name not in attendus:
        f.unlink()

clips, t0, repris = [], time.time(), 0
try:
    for n, phrase in enumerate(phrases, 1):
        chemin = CACHE / nom_cache(n, phrase)
        if chemin.is_file():
            audio, taux = sf.read(chemin, dtype="float32")
            repris += 1
        else:
            torch.manual_seed(graine(n))
            wavs, taux = modele.generate_voice_clone(text=phrase, language="French",
                                                     ref_audio=str(REF_WAV), ref_text=TEXTE_REF)
            audio, taux = wavs[0].astype(np.float32), int(taux)
            tmp = chemin.with_suffix(".tmp")
            sf.write(tmp, audio, taux, format="WAV")
            os.replace(tmp, chemin)
        clips.append((audio, taux))
        print(f"   [{n}/{len(phrases)}] {len(audio)/taux:4.1f} s  {phrase}")
except Exception as e:
    ecrire_etat(statut="echec", erreur=f"synthèse, phrase {n} : {e}")
    raise
print(f"✓ Synthèse : {time.time()-t0:.0f} s ({repris} phrase(s) reprises du cache)")

In [ ]:
# ── Cellule 5 : assemblage, horodatage, contrôle ────────────────────────────
TAUX = clips[0][1]
silence = np.zeros(int(TAUX * PAUSE_MS / 1000), dtype=np.float32)
morceaux, bornes, curseur = [], [], 0
for n, ((audio, taux), phrase) in enumerate(zip(clips, phrases), 1):
    if taux != TAUX:
        audio = librosa.resample(audio, orig_sr=taux, target_sr=TAUX)
    if n > 1:
        morceaux.append(silence); curseur += len(silence)
    debut = curseur
    morceaux.append(audio); curseur += len(audio)
    bornes.append({"n": n, "texte": phrase, "debut_s": round(debut / TAUX, 3), "fin_s": round(curseur / TAUX, 3)})
voix = np.concatenate(morceaux)
import warnings
volume = pyln.Meter(TAUX).integrated_loudness(voix)
if np.isfinite(volume):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # les pics sont traités juste en dessous
        voix = pyln.normalize.loudness(voix, volume, CIBLE_LUFS)
crete = float(np.max(np.abs(voix)))
if crete > 0.95:  # plutôt que d'écrêter (saturation), on baisse un peu tout le volume
    voix = voix * (0.95 / crete)
    print(f"   Pics ramenés sous la saturation (volume baissé de {20 * np.log10(crete / 0.95):.1f} dB)")
voix = voix.astype(np.float32)
sf.write(DOSSIER_VOIX / "voix.wav", voix, TAUX)
duree = len(voix) / TAUX
print(f"✓ voix.wav : {int(duree // 60)} min {int(duree % 60):02d} s")

# Horodatage mot à mot, et contrôle phrase par phrase : on retranscrit la voix et on la compare au texte.
# L'audio est passé en mémoire (16 kHz) : faster-whisper ne lit pas le fichier, ce qui évite
# l'incompatibilité de sa lecture de fichiers avec certaines versions de PyAV (constatée sur Colab le 30/09).
from faster_whisper import WhisperModel
whisper = WhisperModel("large-v3-turbo", device="cuda" if cuda else "cpu", compute_type="float16" if cuda else "int8")
voix_16k = librosa.resample(voix, orig_sr=TAUX, target_sr=16000).astype(np.float32)
segments, _ = whisper.transcribe(voix_16k, language="fr", word_timestamps=True, vad_filter=False)
mots = [{"mot": w.word.strip(), "debut_s": round(w.start, 3), "fin_s": round(w.end, 3)}
        for s in segments for w in (s.words or [])]

# Whisper écrit les nombres en chiffres (« 1344 », « 42 % ») alors que le script les écrit en lettres :
# sans cette conversion, 19 phrases sur 140 du sel étaient signalées à tort (mesuré le 30/09).
UNITES = "zéro un deux trois quatre cinq six sept huit neuf dix onze douze treize quatorze quinze seize".split()
DIZAINES = {1: "dix", 2: "vingt", 3: "trente", 4: "quarante", 5: "cinquante", 6: "soixante", 8: "quatre-vingt"}

def en_lettres(n):
    """Un entier (jusqu'à 999 999 999) en toutes lettres, à la française."""
    if n < 17:
        return UNITES[n]
    if n < 100:
        d, u = divmod(n, 10)
        if d in (7, 9):  # soixante-dix…, quatre-vingt-dix…
            return DIZAINES[d - 1] + ("-et-" if n == 71 else "-") + en_lettres(10 + u)
        if u == 0:
            return DIZAINES[d] + ("s" if d == 8 else "")
        return DIZAINES[d] + ("-et-un" if u == 1 and d != 8 else "-" + UNITES[u])
    if n < 1000:
        c, r = divmod(n, 100)
        tete = "cent" if c == 1 else UNITES[c] + " cent"
        return tete + (" " + en_lettres(r) if r else ("s" if c > 1 else ""))
    for taille, nom in ((10**6, "million"), (1000, "mille")):
        if n >= taille:
            q, r = divmod(n, taille)
            tete = nom if (q == 1 and taille == 1000) else en_lettres(q) + " " + nom + ("s" if taille > 1000 and q > 1 else "")
            return tete + (" " + en_lettres(r) if r else "")

def chiffres_en_lettres(t):
    """Ce que Whisper écrit en chiffres (« 1344 », « 42 % », « 21 km », « 1er »), redit en lettres."""
    t = re.sub(r"(\d)[\s\u202f\u00a0](?=\d{3}\b)", r"\1", t)  # 8 000 → 8000
    t = re.sub(r"\b1er\b", "premier", t)
    t = re.sub(r"\s*%", " pour cent", t)
    t = re.sub(r"\b(\d+)\s*km\b", r"\1 kilomètres", t)
    return re.sub(r"\d+", lambda m: en_lettres(int(m.group())), t)

def lettres(t):
    t = unicodedata.normalize("NFKD", t.lower())
    return re.sub(r"[^a-z]", "", "".join(c for c in t if not unicodedata.combining(c)))

a_reecouter = []
for b in bornes:
    entendu = " ".join(m["mot"] for m in mots if b["debut_s"] - 0.15 <= (m["debut_s"] + m["fin_s"]) / 2 <= b["fin_s"] + 0.15)
    b["entendu"] = entendu
    b["ressemblance"] = round(difflib.SequenceMatcher(None, lettres(b["texte"]), lettres(chiffres_en_lettres(entendu))).ratio(), 2)
    if b["ressemblance"] < SEUIL_A_REECOUTER:
        a_reecouter.append(b["n"])

(DOSSIER_VOIX / "mots.json").write_text(json.dumps(
    {"duree_s": round(duree, 3), "taux_hz": TAUX, "pause_ms": PAUSE_MS, "phrases": bornes, "mots": mots},
    ensure_ascii=False, indent=2), encoding="utf-8")

nb_mots = sum(len(re.findall(r"[\w’'-]+", p)) for p in phrases)
parle = sum(b["fin_s"] - b["debut_s"] for b in bornes)
ecrire_etat(statut="pret", fin=maintenant(), duree_s=round(duree, 1), mots_dits=nb_mots,
            debit_mots_s=round(nb_mots / parle, 2), a_reecouter=a_reecouter, erreur=None)

print(f"✓ mots.json : {len(bornes)} phrases, {len(mots)} mots horodatés")
print(f"✓ Débit mesuré : {nb_mots / parle:.2f} mots/s (à reporter dans Memoire/lecons.md — la skill le fera)")
if a_reecouter:
    print(f"\n⚠️  {len(a_reecouter)} phrase(s) à réécouter (peut-être mal dites) : {a_reecouter}")
    for b in bornes:
        if b["n"] in a_reecouter:
            print(f"   [{b['n']}] {b['debut_s']:.1f} s — écrit : {b['texte'][:60]}\n{'':10}entendu : {b['entendu'][:60]}")
    print("   → Si l'une est vraiment mal dite : mets son numéro dans REFAIRE_LIGNES et relance tout.")
print(f"\n✅ Voix prête dans Videos/{DOSSIER.name}/voix/")